In [1]:
import re
import pandas as pd
from scipy import stats
import re
import pandas as pd
from pathlib import Path
from IPython.display import display

In [2]:
def normalize_text(text):
    """Normalize text: NFKD, remove accents, lowercase, remove parentheses content"""
    return text.str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8').str.lower().apply(lambda x: re.sub(r'\s*\([^)]*\)', '', x))


def load_and_prepare_data(interdis_path, institutions_path, label_col_name):
    """
    Load interdis CSV, rename Unnamed: 0 to ranking_position, 
    merge with institutions names, and normalize institution names.
    
    Args:
        interdis_path: path to sorted_interdis.csv
        institutions_path: path to institutions_names.csv
        label_col_name: column name to use for normalized names (e.g., 'LABEL' or 'normalized_Universidade')
    
    Returns:
        DataFrame with ranking_position, LABEL, and normalized_name columns
    """
    # Load interdis data and rename index column
    df = pd.read_csv(interdis_path, encoding="utf-8")
    df.rename(columns={'Unnamed: 0': 'ranking_position'}, inplace=True)
    
    # Load and merge institutions names
    institutions = pd.read_csv(institutions_path, encoding="utf-8")
    df['LABEL'] = institutions['ins_name']
    
    # Normalize institution names
    df['normalized_name'] = normalize_text(df['LABEL'])
    
    return df

def load_all_subfolders(parent_folder):
    """
    Load and prepare data from all subfolders containing sorted_interdis.csv and institutions_names.csv
    
    Args:
        parent_folder: path to parent folder (e.g., './data/macro' or './data/sub')
    
    Returns:
        Dictionary with subfolder names as keys and prepared DataFrames as values
    """
    data = {}
    parent_path = Path(parent_folder)
    
    # Find all subdirectories with sorted_interdis.csv
    for subfolder in sorted(parent_path.glob('*')):
        if subfolder.is_dir():
            interdis_file = subfolder / 'sorted_interdis.csv'
            institutions_file = subfolder / 'institutions_names.csv'
            print(interdis_file)
            print(institutions_file)
            print()
            
            if interdis_file.exists() and institutions_file.exists():
                try:
                    df = load_and_prepare_data(
                        str(interdis_file),
                        str(institutions_file),
                        'LABEL'
                    )
                    data[subfolder.name] = df
                    print(f"✓ Loaded {subfolder.name}: {len(df)} rows")
                except Exception as e:
                    print(f"✗ Error loading {subfolder.name}: {e}")
    print(data.keys())
    return data

In [3]:
print("Loading macro subfolders:")
macro_data = load_all_subfolders('./data/macro')

print("\nLoading sub subfolders:")
sub_data = load_all_subfolders('./data/sub')

ruf = pd.read_csv('./data/RUF.csv', encoding="utf-8", delimiter=';')
ruf['normalized_Universidade'] = normalize_text(ruf['Universidade'])

print(f"\n✓ Loaded {len(macro_data)} macro subfolders and {len(sub_data)} sub subfolders")

Loading macro subfolders:
data\macro\macro_no_quality\sorted_interdis.csv
data\macro\macro_no_quality\institutions_names.csv

✓ Loaded macro_no_quality: 200 rows
data\macro\macro_percentile\sorted_interdis.csv
data\macro\macro_percentile\institutions_names.csv

✓ Loaded macro_percentile: 200 rows
data\macro\macro_q1\sorted_interdis.csv
data\macro\macro_q1\institutions_names.csv

✓ Loaded macro_q1: 200 rows
data\macro\macro_q2\sorted_interdis.csv
data\macro\macro_q2\institutions_names.csv

✓ Loaded macro_q2: 200 rows
data\macro\macro_q3\sorted_interdis.csv
data\macro\macro_q3\institutions_names.csv

✓ Loaded macro_q3: 200 rows
data\macro\macro_q4\sorted_interdis.csv
data\macro\macro_q4\institutions_names.csv

✓ Loaded macro_q4: 200 rows
data\macro\macro_quartile\sorted_interdis.csv
data\macro\macro_quartile\institutions_names.csv

✓ Loaded macro_quartile: 200 rows
dict_keys(['macro_no_quality', 'macro_percentile', 'macro_q1', 'macro_q2', 'macro_q3', 'macro_q4', 'macro_quartile'])

Loadi

In [4]:
# Compute Kendall Tau for each macro subfolder
print("=== MACRO KENDALL TAU RESULTS ===\n")
macro_kendall_results = {}

for name, merged in macro_data.items():
    merged['Ranking_num'] = pd.to_numeric(merged['Ranking'], errors='coerce')
    merged['DIV_STAR'] = pd.to_numeric(merged['ranking_position'], errors='coerce')
    
    mask = merged['Ranking_num'].notna() & merged['ranking_position'].notna()
    x = merged.loc[mask, 'Ranking_num']
    y = merged.loc[mask, 'ranking_position']
    
    print(f"{name}:")
    print(f"  Paired observations: {len(x)}")
    
    if len(x) >= 2:
        tau, p_value = stats.kendalltau(x, y, nan_policy='omit')
        macro_kendall_results[name] = {'tau': tau, 'p_value': p_value, 'n': len(x)}
        print(f"  Kendall Tau: {tau:.6f}")
        print(f"  p-value: {p_value}")
    else:
        macro_kendall_results[name] = {'tau': None, 'p_value': None, 'n': len(x)}
        print(f"  Not enough paired observations (need >=2)")
    print()


=== MACRO KENDALL TAU RESULTS ===

macro_no_quality:
  Paired observations: 135
  Kendall Tau: 0.428548
  p-value: 1.6702310091935617e-13

macro_percentile:
  Paired observations: 136
  Kendall Tau: 0.420829
  p-value: 3.6618021175797793e-13

macro_q1:
  Paired observations: 137
  Kendall Tau: 0.560893
  p-value: 2.410541257140024e-22

macro_q2:
  Paired observations: 135
  Kendall Tau: 0.514346
  p-value: 8.835665651195046e-19

macro_q3:
  Paired observations: 138
  Kendall Tau: 0.562179
  p-value: 1.3518910510553248e-22

macro_q4:
  Paired observations: 133
  Kendall Tau: 0.566334
  p-value: 4.0818465582274343e-22

macro_quartile:
  Paired observations: 135
  Kendall Tau: 0.437172
  p-value: 5.42690207332743e-14



In [5]:
# Compute Kendall Tau for each sub subfolder
print("=== SUB KENDALL TAU RESULTS ===\n")
sub_kendall_results = {}

for name, merged in sub_data.items():
    merged['Ranking_num'] = pd.to_numeric(merged['Ranking'], errors='coerce')
    merged['ranking_position'] = pd.to_numeric(merged['ranking_position'], errors='coerce')
    
    mask = merged['Ranking_num'].notna() & merged['ranking_position'].notna()
    x = merged.loc[mask, 'Ranking_num']
    y = merged.loc[mask, 'ranking_position']
    
    print(f"{name}:")
    print(f"  Paired observations: {len(x)}")
    
    if len(x) >= 2:
        tau, p_value = stats.kendalltau(x, y, nan_policy='omit')
        sub_kendall_results[name] = {'tau': tau, 'p_value': p_value, 'n': len(x)}
        print(f"  Kendall Tau: {tau:.6f}")
        print(f"  p-value: {p_value}")
    else:
        sub_kendall_results[name] = {'tau': None, 'p_value': None, 'n': len(x)}
        print(f"  Not enough paired observations (need >=2)")
    print()


=== SUB KENDALL TAU RESULTS ===

sub_no_quality:
  Paired observations: 139
  Kendall Tau: 0.615401
  p-value: 6.10766830919195e-27

sub_percentile:
  Paired observations: 138
  Kendall Tau: 0.609997
  p-value: 2.5790231500270442e-26

sub_q1:
  Paired observations: 136
  Kendall Tau: 0.685985
  p-value: 2.2404408886025156e-32

sub_q2:
  Paired observations: 138
  Kendall Tau: 0.652314
  p-value: 7.434778922699925e-30

sub_q3:
  Paired observations: 137
  Kendall Tau: 0.628952
  p-value: 1.120538516406508e-27

sub_q4:
  Paired observations: 135
  Kendall Tau: 0.566753
  p-value: 1.8343517843642197e-22

sub_quartile:
  Paired observations: 139
  Kendall Tau: 0.606851
  p-value: 3.0474799448935324e-26



In [6]:
summary_rows = []

def process_merged_dict(merged_dict, group_type):
    for name, merged in merged_dict.items():
        df = merged.copy()
        # Ensure numeric columns exist
        df['Ranking_num'] = pd.to_numeric(df.get('Ranking', df.get('Ranking_num')), errors='coerce')
        df['ranking_position_num'] = pd.to_numeric(df.get('ranking_position'), errors='coerce')

        mask = df['Ranking_num'].notna() & df['ranking_position_num'].notna()
        paired = df.loc[mask, ['LABEL', 'normalized_name', 'Ranking_num', 'ranking_position_num']].copy()
        paired.rename(columns={'Ranking_num': 'ruf_ranking', 'ranking_position_num': 'merged_ranking'}, inplace=True)
        paired['source'] = f'{group_type}/{name}'

        # Compute group-level Kendall Tau if we have enough pairs
        if len(paired) >= 2:
            tau, p_value = stats.kendalltau(paired['ruf_ranking'], paired['merged_ranking'], nan_policy='omit')
        else:
            tau, p_value = (None, None)
            
        summary_rows.append({'source': f'{group_type}/{name}', 'tau': tau, 'p_value': p_value, 'n': len(paired)})

# Process macro and sub merged dictionaries created earlier in the notebook
process_merged_dict(macro_data, 'macro')
process_merged_dict(sub_data, 'sub')

kendall_summary = pd.DataFrame(summary_rows)

# Display concise summary and first rows of paired table
print('=== Kendall summary (per folder) ===')
display(kendall_summary.sort_values('source').reset_index(drop=True))

print(f'✓ Saved kendall summary: {len(kendall_summary)} rows to ./data/kendall_summary.csv')
kendall_summary.to_csv('./data/kendall_summary.csv', index=False, encoding='utf-8')

=== Kendall summary (per folder) ===


,source,tau,p_value,n
0,macro/macro_no_quality,0.428548,1.670231e-13,135
1,macro/macro_percentile,0.420829,3.661802e-13,136
2,macro/macro_q1,0.560893,2.410541e-22,137
3,macro/macro_q2,0.514346,8.835666e-19,135
4,macro/macro_q3,0.562179,1.351891e-22,138
5,macro/macro_q4,0.566334,4.081847e-22,133
6,macro/macro_quartile,0.437172,5.426902e-14,135
7,sub/sub_no_quality,0.615401,6.107668e-27,139
8,sub/sub_percentile,0.609997,2.579023e-26,138
9,sub/sub_q1,0.685985,2.240441e-32,136


✓ Saved kendall summary: 14 rows to ./data/kendall_summary.csv
